# 🏠 Analizador de Flipping Inmobiliario — Buenos Aires

Este notebook busca oportunidades de flipping en **ZonaProp** y **MercadoLibre**,
analiza precio por m², ventajas/desventajas con IA y genera un reporte Excel descargable.

**Cómo usarlo:**
1. Ejecutá la celda de instalación (▶)
2. Ingresá tu API key de Anthropic
3. Elegí barrio y fuentes
4. Ejecutá el análisis
5. Descargá el Excel

> 💡 Podés ejecutar todas las celdas de una vez con **Runtime → Run all**

In [ ]:
#@title ⚙️ Paso 1: Instalación (ejecutar una sola vez)
import subprocess, sys

print("📦 Instalando dependencias...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "anthropic>=0.40.0", "httpx>=0.27.0", "beautifulsoup4>=4.12.0",
    "rich>=13.7.0", "python-dotenv>=1.0.0", "lxml>=5.2.0",
    "tenacity>=8.3.0", "openpyxl>=3.1.0"
], check=True)

print("📥 Descargando código del repositorio...")
import os
if not os.path.exists("real_estate_agent"):
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/imajul/real_estate_agent.git"], check=True)
else:
    subprocess.run(["git", "-C", "real_estate_agent", "pull"], check=True)

os.chdir("real_estate_agent")
if str(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.getcwd())

print("✅ Listo!")

In [ ]:
#@title 🔑 Paso 2: Configuración
import os
from getpass import getpass

#@markdown ### API Key de Anthropic
#@markdown Obtené tu key en [console.anthropic.com](https://console.anthropic.com)
ANTHROPIC_API_KEY = "" #@param {type:"string"}

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Pegá tu Anthropic API Key (no se muestra): ")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

#@markdown ---
#@markdown ### Barrio a analizar
BARRIO = "palermo" #@param ["palermo", "belgrano", "recoleta", "villa_crespo", "caballito", "flores", "almagro", "san_telmo", "nunez", "colegiales", "chacarita", "boedo", "villa_urquiza"]

#@markdown ### Fuentes
USAR_ZONAPROP = True #@param {type:"boolean"}
USAR_MERCADOLIBRE = True #@param {type:"boolean"}

#@markdown ### Cantidad de propiedades a rankear
TOP_N = 10 #@param {type:"slider", min:3, max:20, step:1}

#@markdown ### Modo demo (datos de ejemplo, sin internet)
MODO_DEMO = True #@param {type:"boolean"}

sources = []
if USAR_ZONAPROP: sources.append("zonaprop")
if USAR_MERCADOLIBRE: sources.append("mercadolibre")
if not sources: sources = ["zonaprop", "mercadolibre"]

print(f"✅ Configuración lista:")
print(f"   Barrio: {BARRIO}")
print(f"   Fuentes: {', '.join(sources)}")
print(f"   Top N: {TOP_N}")
print(f"   Modo demo: {MODO_DEMO}")
print(f"   API Key: {'configurada ✓' if ANTHROPIC_API_KEY else '⚠️ no configurada'}")

In [ ]:
#@title 🔍 Paso 3: Ejecutar análisis
import importlib, sys

# Reload modules in case of re-run
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["models","config","demo_data","scrapers","analyzers","reports"]):
        del sys.modules[mod]

from models import Property
from analyzers.market import compute_market_reference
from analyzers.ai_evaluator import AIEvaluator, batch_evaluate

# ── Recolección de propiedades ─────────────────────────────────────────────
all_properties = []

if MODO_DEMO:
    print("📦 Cargando propiedades de demo...")
    from demo_data import DEMO_PROPERTIES
    nb = BARRIO.lower()
    all_properties = [p for p in DEMO_PROPERTIES if nb in p.neighborhood.lower()]
    if not all_properties:
        all_properties = list(DEMO_PROPERTIES)
    print(f"   → {len(all_properties)} propiedades cargadas")
else:
    if "zonaprop" in sources:
        print("🌐 Scrapeando ZonaProp...")
        try:
            from scrapers.zonaprop import ZonaPropScraper
            with ZonaPropScraper() as s:
                props = s.search(BARRIO, 50)
            all_properties.extend(props)
            print(f"   → {len(props)} propiedades de ZonaProp")
        except Exception as e:
            print(f"   ⚠️ ZonaProp error: {e}")

    if "mercadolibre" in sources:
        print("🌐 Scrapeando MercadoLibre...")
        try:
            from scrapers.mercadolibre import MercadoLibreScraper
            with MercadoLibreScraper() as s:
                props = s.search(BARRIO, 50)
            all_properties.extend(props)
            print(f"   → {len(props)} propiedades de MercadoLibre")
        except Exception as e:
            print(f"   ⚠️ MercadoLibre error: {e}")

if not all_properties:
    raise RuntimeError("No se encontraron propiedades. Activá Modo Demo o verificá la conexión.")

# ── Análisis de mercado ────────────────────────────────────────────────────
display_nb = BARRIO if not MODO_DEMO or BARRIO != "all" else "Buenos Aires"
print(f"\n📊 Calculando referencia de mercado para {display_nb}...")
market_ref = compute_market_reference(all_properties, display_nb)
print(f"   Precio mediana: USD {market_ref.median_price_per_m2_usd:,.0f}/m²")
print(f"   Muestra: {market_ref.sample_count}/{len(all_properties)} propiedades con precio/m²")

# ── Evaluación con IA ──────────────────────────────────────────────────────
evaluator = AIEvaluator()
if evaluator.available:
    print(f"\n🤖 Analizando top {TOP_N} propiedades con Claude AI...")
    print("   (esto puede tardar 1-2 minutos)")
else:
    print(f"\n📐 Analizando top {TOP_N} propiedades con análisis heurístico...")

analyses = batch_evaluate(all_properties, market_ref, evaluator, top_n=TOP_N)
print(f"\n✅ Análisis completo — {len(analyses)} propiedades evaluadas")

In [ ]:
#@title 📊 Paso 4: Ver resultados
import pandas as pd
from IPython.display import display, HTML

rows = []
for i, a in enumerate(analyses, 1):
    p = a.property
    surface = p.surface_covered or p.surface_total
    rows.append({
        "#": i,
        "Recomendación": a.flipping_recommendation or "—",
        "Score": f"{a.flipping_score:.1f}/10",
        "Barrio": p.neighborhood,
        "Título": p.title[:50],
        "Precio": p.display_price,
        "m²": f"{surface:.0f}" if surface else "—",
        "USD/m²": f"USD {p.price_per_m2:,.0f}" if p.price_per_m2 else "—",
        "vs. mercado": f"{a.discount_vs_market_pct:+.1f}%" if a.discount_vs_market_pct else "—",
        "Ganancia est.": f"USD {a.estimated_profit_usd:,.0f}" if a.estimated_profit_usd else "—",
        "ROI": f"{a.roi_pct:.0f}%" if a.roi_pct else "—",
        "Fuente": p.source.value,
    })

df = pd.DataFrame(rows)

def color_rec(val):
    if val == "COMPRAR":     return "background-color: #C6EFCE; color: #276221; font-weight: bold"
    if val == "ANALIZAR MÁS": return "background-color: #FFEB9C; color: #9C6500; font-weight: bold"
    if val == "DESCARTAR":  return "background-color: #FFC7CE; color: #9C0006; font-weight: bold"
    return ""

def color_disc(val):
    try:
        v = float(val.replace("%","").replace("+",""))
        if v <= -10: return "color: #276221; font-weight: bold"
        if v >= 10:  return "color: #9C0006; font-weight: bold"
    except: pass
    return ""

styled = (df.style
    .applymap(color_rec, subset=["Recomendación"])
    .applymap(color_disc, subset=["vs. mercado"])
    .set_properties(**{"text-align": "left", "font-size": "13px"})
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color", "#1F3864"), ("color", "white"),
                  ("font-weight", "bold"), ("padding", "8px")]
    }])
    .hide(axis="index")
)

display(HTML(f"<h3>📊 Mercado: {display_nb} — Precio mediana USD {market_ref.median_price_per_m2_usd:,.0f}/m²</h3>"))
display(styled)

In [ ]:
#@title 🔎 Paso 5: Detalle de las mejores oportunidades
from IPython.display import display, HTML

MOSTRAR_TOP = 3 #@param {type:"slider", min:1, max:10, step:1}

for i, a in enumerate(analyses[:MOSTRAR_TOP], 1):
    p = a.property
    surface = p.surface_covered or p.surface_total

    rec_color = {"COMPRAR": "#C6EFCE", "ANALIZAR MÁS": "#FFEB9C", "DESCARTAR": "#FFC7CE"}.get(a.flipping_recommendation, "#f0f0f0")
    rec_text  = {"COMPRAR": "#276221", "ANALIZAR MÁS": "#9C6500", "DESCARTAR": "#9C0006"}.get(a.flipping_recommendation, "#000")

    pros_html = "".join(f"<li>✅ {x}</li>" for x in a.pros) if a.pros else "<li>Sin datos</li>"
    cons_html = "".join(f"<li>❌ {x}</li>" for x in a.cons) if a.cons else "<li>Sin datos</li>"
    reno_html = "".join(f"<li>🔧 {x}</li>" for x in a.renovation_suggestions) if a.renovation_suggestions else "<li>Sin datos</li>"

    discount_str = f"{a.discount_vs_market_pct:+.1f}%" if a.discount_vs_market_pct is not None else "—"
    profit_str   = f"USD {a.estimated_profit_usd:,.0f}" if a.estimated_profit_usd else "—"
    roi_str      = f"{a.roi_pct:.0f}%" if a.roi_pct else "—"
    arv_str      = f"USD {a.estimated_arv_usd:,.0f}" if a.estimated_arv_usd else "—"
    reno_str     = f"USD {a.estimated_renovation_cost_usd:,.0f}" if a.estimated_renovation_cost_usd else "—"

    html = f"""
    <div style="border:1px solid #ccc; border-radius:8px; padding:16px; margin:12px 0; font-family:sans-serif">
      <h3 style="margin:0 0 8px">#{i} {p.title}</h3>
      <span style="background:{rec_color}; color:{rec_text}; font-weight:bold;
                   padding:4px 12px; border-radius:4px; font-size:14px">
        {a.flipping_recommendation} — Score: {a.flipping_score:.1f}/10
      </span>

      <table style="width:100%; margin-top:12px; border-collapse:collapse; font-size:13px">
        <tr>
          <td style="padding:4px 8px"><b>Barrio:</b> {p.neighborhood}</td>
          <td style="padding:4px 8px"><b>Precio:</b> {p.display_price}</td>
          <td style="padding:4px 8px"><b>Superficie:</b> {f"{surface:.0f} m²" if surface else "—"}</td>
          <td style="padding:4px 8px"><b>USD/m²:</b> {f"USD {p.price_per_m2:,.0f}" if p.price_per_m2 else "—"}</td>
        </tr>
        <tr>
          <td style="padding:4px 8px"><b>Desc. mercado:</b> {discount_str}</td>
          <td style="padding:4px 8px"><b>ARV estimado:</b> {arv_str}</td>
          <td style="padding:4px 8px"><b>Costo reno.:</b> {reno_str}</td>
          <td style="padding:4px 8px"><b>Ganancia / ROI:</b> {profit_str} / {roi_str}</td>
        </tr>
        <tr>
          <td colspan="4" style="padding:4px 8px">
            <b>Dirección:</b> {p.address} &nbsp;|&nbsp;
            <b>Amenities:</b> {', '.join(p.amenities) if p.amenities else '—'} &nbsp;|&nbsp;
            <a href="{p.url}" target="_blank">Ver aviso ↗</a>
          </td>
        </tr>
      </table>

      <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px; margin-top:12px">
        <div>
          <b>✅ Ventajas</b>
          <ul style="margin:4px 0; padding-left:16px; font-size:12px">{pros_html}</ul>
        </div>
        <div>
          <b>❌ Desventajas</b>
          <ul style="margin:4px 0; padding-left:16px; font-size:12px">{cons_html}</ul>
        </div>
        <div>
          <b>🔧 Renovación sugerida</b>
          <ul style="margin:4px 0; padding-left:16px; font-size:12px">{reno_html}</ul>
        </div>
      </div>

      <div style="margin-top:10px; padding:8px; background:#f8f9fa; border-radius:4px; font-size:12px">
        <b>💬 Resumen IA:</b> {a.ai_summary}
      </div>
    </div>
    """
    display(HTML(html))

In [ ]:
#@title 📥 Paso 6: Descargar Excel
from reports.excel import export_excel
from google.colab import files
from datetime import datetime

nombre_archivo = f"flipping_{display_nb.lower().replace(' ','_')}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

path = export_excel(
    analyses=analyses,
    market_ref=market_ref,
    neighborhood=display_nb,
    total_props=len(all_properties),
    output_path=nombre_archivo,
)

print(f"✅ Excel generado: {path}")
print("📥 Iniciando descarga...")
files.download(path)